# 01 · The n-tables, one site: every method under one fixed configuration

**The question.** Given the same images, the same pixels and the same water level, how close does each
elevation method come to the surveyed bed, and how does that change with the number of images n? One
configuration, external to every method and identical to MAREA's production one, is given to all of them:

| input | the production rule |
|---|---|
| scenes | transition-zone cloud screening (`pit.reference_and_clouds`, then `pit.filter_dates(cloud_threshold=0.10)`), with an overpass time and a finite boundary level |
| per-pixel mask | clear = SCL in {4, 5, 6, 7} |
| index | NDWI = (B03 − B08) / (B03 + B08); wet = NDWI > 0 |
| pixels | the production extraction's pixels (`marea_extract.npz`) |
| water level | the production boundary `make_boundary(aoi, "EOT20", n_gauges, period, exclude=judges)`; MAREA reads it τ minutes earlier, with its band lags refitted on each window; every other method reads it without lag |
| window | the n most recent screened images on or before the truth date, n = 125, 100, 75, 60, 50, 40 |

**The truth.** Dutch sites: the Rijkswaterstaat Vaklodingen, only the cells surveyed in 2023–2025, each with
its most recent survey in that range, dated to the end of its survey year. At the Wadden the pixels are grouped
by that year (2023, 2024, 2025) and each group is a row with its own windows. Villaviciosa: the PNOA LiDAR,
pending the user's decision; without it the notebook says *pending truth* and stops. The RTK survey is never
read.

**The methods**, each a plain function call on the same window's scenes, timed with `time.perf_counter`:
MAREA; NIDEM (`pyintertidal.waterline_contour`, with the common mask); DEA Intertidal 2.5.3
(`experiments/fixedcfg/dea_intertidal.py`, DEA's own code in a side interpreter); LogiTide2DEM (E4),
Flood2Topo (E5a), Tseng linear scaling (E5b), sampled exceedance (E5c), Coast X-Ray (E7) and the wet/dry
bracket (E12) from `experiments/fixedcfg/methods.py`.

**Outputs.** `results/fixedcfg_nb/<site>.csv`, one row per n and method (per survey-year group and n at the
Wadden) with the scores and the seconds, and the error, bias and coverage curves against n.
`01b_fixed_config_n_tables_summary.ipynb` gathers the sites. The same design is scripted in
`experiments/v19_fixed_config_tables.py`; the last cell compares with its outputs when they exist.

## Parameters

Everything that changes between runs lives in the next cell. The environment variables `NB_SITE`, `NB_NS`,
`NB_METHODS` and `NB_EXTRA` override the first four, `NB_OUT` the output folder (the server script
sets `NB_SITE`). Each site is described
as its production notebook runs it (`tide_boundary_comparison_<site>.ipynb`,
`intertidal_topography_villaviciosa.ipynb`).

In [ ]:
import os

SITE    = os.environ.get("NB_SITE", "escalda")       # escalda | wadden | ems | villaviciosa
NS      = [int(v) for v in os.environ.get("NB_NS", "125,100,75,60,50,40").split(",")]
METHODS = os.environ.get("NB_METHODS", "MAREA,NIDEM,DEA,E4,E5a,E5b,E5c,E7,E12").split(",")
EXTRA   = os.environ.get("NB_EXTRA", "auto")          # pre-2023 scenes: "auto" reads
                                                      # results/fixedcfg/extra/<site>_extract_<years>.npz
                                                      # when it exists; "none"; or a path

# ── Villaviciosa's truth: the PNOA LiDAR, pending the user's decision ────
VVA_LIDAR      = os.environ.get("NB_VVA_LIDAR")       # a GeoTIFF DTM, e.g. data_v4/truth/pnoa/MDT02-WGS84-0015-3-COB2.tif
VVA_LIDAR_DATE = os.environ.get("NB_VVA_LIDAR_DATE")  # its flight date YYYY-MM-DD: the windows end there

# ── Fixed by the design ─────────────────────────────────────────────────
MAX_PX            = 200_000   # a larger F is a seeded random subsample (v19's rule and seed)
SEED              = 20261019
RANGE_PAD_M       = 0.25      # truth kept inside the production no-delay range ± 0.25 m
MAREA_MIN_BRACKET = 0         # 0 = the rule of the published 10 m products (v19's headline);
                              # marea.QUALITY_MIN_BRACKET = 3 is the current production rule
OUT     = os.environ.get("NB_OUT", "results/fixedcfg_nb")    # the CSV and figures go here
V19_OUT = "results/fixedcfg_truthdate"                # v19's outputs, compared in the last cell
DEA_TMP = os.environ.get("MAREA_DEA_TMP")             # DEA's temp root, outside the repository (default below)

In [ ]:
# ── Each site as its production notebook runs it ────────────────────────
def dutch(site, n_gauges, judges, mouth_side, truth_years):
    d, tag = f"products_{site}_10m", f"eot20_g{n_gauges}"
    return dict(cache=f"ndwi_cube_{site}_2023-2025_10m.nc", extract=f"{d}/marea_extract.npz",
                overpass=f"{d}/overpass_times.json", marea=f"{d}/marea_{tag}",
                nodelay=(f"{d}/comparison_{tag}/products.npz", "z_plain"),
                period=("2023-01-01", "2025-12-31"), n_gauges=n_gauges, judges=judges,
                mouth_side=mouth_side, truth_years=truth_years)


SITES = {
    "escalda": dutch("escalda", 2, ["trnz"], "west", [2025]),
    "wadden":  dutch("wadden", 1, ["harl"], "north+west", [2025, 2024, 2023]),
    "ems":     dutch("ems", 1, ["delf"], "north+west", [2025]),
    "villaviciosa": dict(cache="ndwi_cube_villaviciosa_grande_10y.nc", extract="products_marea/marea_extract.npz",
                         overpass="products_marea/overpass_times.json",
                         marea="products_marea/marea_eot20_g0_2023-2025",
                         nodelay=("products_marea/external_products.npz", "no_delay"),
                         period=("2016-01-01", "2025-12-31"), n_gauges=None, judges=[], mouth_side="any",
                         truth_years=None),
}
P = SITES[SITE]
print(f"{SITE}: n = {NS}; methods {METHODS}")
print(f"  cube {P['cache']} · extraction {P['extract']} · boundary EOT20 + {P['n_gauges'] or 0} gauge(s), "
      f"held out {P['judges']}")

## 0 · Setup

Every path below is relative to the repository root; opened from `notebooks/validation/`, the first lines
move there. No Copernicus connection is needed: the cube, the extraction, the overpass times and the gauge
records are cached. DEA's temp root defaults to `v19_tmp` next to the repository (the server's `TMPDIR` lies
inside the repository, which the DEA adapter refuses).

In [ ]:
import sys
while not os.path.isdir("pyintertidal") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")                                    # from notebooks/validation/ to the repository root
sys.path.insert(0, os.getcwd())

import glob, json, time, types, zlib
from collections import Counter
import numpy as np
import pandas as pd
import xarray as xr
import yaml
import matplotlib.pyplot as plt

import pyintertidal as pit
from pyintertidal import SentinelCube, marea
from pyintertidal import tide_estimators as te
from pyintertidal import waterline_contour as wc
from pyintertidal.boundary import make_boundary, nearest_gauges
from experiments.fixedcfg import methods as fm
from experiments.fixedcfg import dea_intertidal as dea
from experiments.fixedcfg.elev_base import NotComputable
from experiments.fixedcfg.level_series import LevelSeries

pit.net.use_system_certificates()
os.makedirs(OUT, exist_ok=True)
DEA_TMP = DEA_TMP or os.path.join(os.path.dirname(os.getcwd()), "v19_tmp")
LABELS = {"MAREA": "MAREA", "NIDEM": "NIDEM", "DEA": dea.LABEL, **{m: e[0] for m, e in fm.METHODS.items()}}
print("repository", os.getcwd(), "· pyintertidal", pit.__version__)
print("competitors registered:", list(fm.METHODS), "· DEA temp root", DEA_TMP)

## 1 · Study area, instruments and truth

The polygon comes from the registry. The boundary gauges are the N nearest IOC gauges that are not held out as
judges, as in the production notebook. The truth fixes where every window ends:

* **Dutch sites.** The Vaklodingen gives the survey year only, so a survey is dated to the end of its year.
  Only cells surveyed in 2023–2025 count, each with its most recent survey in that range. At the Wadden (6 of
  15 sheets surveyed in 2025) the pixels are grouped by that year and each group is a row with its own
  windows. At the Westerschelde and the Ems the row is the 2025 survey: there the most recent survey is 2025
  for nearly every pixel, and the others (a few hundred at the Westerschelde, none at the Ems) are counted in
  section 6 and left out (they would need windows of their own; adding their years to `truth_years` makes
  them rows).
* **Villaviciosa.** The PNOA LiDAR, pending the user's decision (`VVA_LIDAR`, `VVA_LIDAR_DATE`). Without it
  the notebook says *pending truth* and stops here.

In [ ]:
aoi = pit.sites.get(SITE)
lat_c, lon_c = aoi.centroid
print(aoi, f"· centroid {lat_c:.3f}, {lon_c:.3f}")
near = nearest_gauges(lat_c, lon_c, 6)
members = [g[0] for g in near if g[0] not in P["judges"]][:(P["n_gauges"] or 0)]
for code, name, km in near:
    role = "boundary" if code in members else "judge (held out)" if code in P["judges"] else "unused"
    print(f"  {code:5s} {name:28s} {km:5.1f} km   {role}")

In [ ]:
if SITE == "villaviciosa":
    if not (VVA_LIDAR and os.path.exists(VVA_LIDAR) and VVA_LIDAR_DATE):
        pending = [dict(site=SITE, n=n, method=m, status="pending truth",
                        reason="the PNOA LiDAR (VVA_LIDAR) and its flight date (VVA_LIDAR_DATE) are not given")
                   for n in NS for m in METHODS]
        pd.DataFrame(pending).to_csv(f"{OUT}/{SITE}.csv", index=False)
        print(f"pending truth: no PNOA LiDAR and flight date given; nothing is scored -> {OUT}/{SITE}.csv")
        raise SystemExit("pending truth")
    ROWS = [dict(site=SITE, year=0, end=VVA_LIDAR_DATE)]
else:
    years = P["truth_years"]
    ROWS = [dict(site=f"{SITE}-{y}" if len(years) > 1 else SITE, year=y, end=f"{y}-12-31") for y in years]
for row in ROWS:
    print(f"row {row['site']:12s} truth dated {row['end']}: its windows end there")

## 2 · The datacube

Cached at 10 m (B03, B08, SCL), as for the production notebook. Every later step reads it from disk.

In [ ]:
cube = SentinelCube(aoi, P["period"], water="ndwi", cache_path=P["cache"], resolution=10)
cube.ensure(None)                                      # cached: no connection needed
transform, crs = cube.grid
H, W = cube.shape
print(f"{len(cube.dates)} scenes {cube.dates[0]} → {cube.dates[-1]} · grid {H}×{W} @ {abs(transform.a):g} m")

## 3 · The production scene screening

The two streaming passes and the parameters of the production notebooks: the reference map voted by the clean
scenes (at most 5 % bad SCL over the frame; stable where 95 % of the votes agree; a 10-pixel halo around the
transition zone), then every scene's cloud share over the transition zone. A scene is usable when at most
10 % of the transition zone is cloudy.

In [ ]:
t0 = time.perf_counter()
reference_map, cloud_pct = pit.reference_and_clouds(
    cube, threshold=0.0, bad_classes=pit.water.BAD_CLASSES, clean_scene_max_bad=0.05,
    stable_threshold=0.95, transition_buffer_px=10)
usable = pit.filter_dates(cloud_pct, cloud_threshold=0.10)
for code, name in [(0, "transition"), (1, "stable water"), (2, "stable land")]:
    print(f"  class {code} ({name:13s}): {int((reference_map == code).sum()):>10,} px")
print(f"{len(usable)} of {len(cloud_pct)} scenes usable  [{time.perf_counter() - t0:.0f} s]")

## 4 · The boundary and the screened list S

One call builds the production boundary (EOT20 and the nearest gauges, their consensus; EOT20 alone at
Villaviciosa). A usable scene enters the screened list S when it has an overpass time and a finite boundary
level there. S's 2023–2025 part must be the record of the production MAREA product (`record_dates` of its
`result.json`).

In [ ]:
boundary = make_boundary(aoi, tide_model="EOT20", n_gauges=P["n_gauges"], period=P["period"],
                         exclude=P["judges"])
print(f"boundary: {boundary.name}")
overpass = {d: pd.Timestamp(v) for d, v in json.load(open(P["overpass"])).items()}
cand = [d for d in usable if d in overpass]
t_cand = pd.DatetimeIndex([overpass[d] for d in cand]).tz_localize(None)
ok = np.isfinite(np.asarray(boundary.levels(t_cand), float))
S_dates, S_t = np.array(cand)[ok], t_cand[ok]
print(f"S: {len(S_dates)} screened scenes {S_dates[0]} → {S_dates[-1]} ({len(usable) - len(cand)} usable "
      f"scenes without an overpass time, {int((~ok).sum())} without a finite level)")

record = [d[:10] for d in json.load(open(f"{P['marea']}/result.json"))["record_dates"]]
part = [d for d in S_dates if "2023-01-01" <= d <= "2025-12-31"]
print(f"S in 2023-2025 is the production MAREA record ({len(record)} scenes): {part == record}")

### Pre-2023 scenes (optional)

The legacy cubes hold 2023–2025 only (68 / 71 / 54 screened scenes at the Westerschelde / Wadden / Ems), so
windows of 75 images or more need earlier years. `experiments/light_to_extraction.py` writes them as an extra
extraction on the same pixels, with the production screening of every scene (`cloud_pct` against the
production reference map) and its overpass time in the sidecar JSON. Their screened scenes go before S; their
levels come from the same boundary object (where the 2023–2025 gauge records have no data, the consensus is
EOT20 alone). NIDEM works on whole frames, which the extra extraction does not hold: a window with pre-2023
scenes has no NIDEM.

In [ ]:
found = sorted(glob.glob(f"results/fixedcfg/extra/{SITE}_extract_*.npz"))
EXTRA_PATH = (found[-1] if found else None) if EXTRA == "auto" else (None if EXTRA == "none" else EXTRA)
x_dates = np.array([], str)
if EXTRA_PATH:
    side = json.load(open(EXTRA_PATH[:-4] + ".json", encoding="utf-8"))
    cloud_x = {d: p for d, p in side["cloud_pct"].items() if p is not None and d < P["period"][0]}
    cand_x = [d for d in pit.filter_dates(cloud_x, cloud_threshold=0.10) if side["overpass"].get(d)]
    t_x = pd.DatetimeIndex([pd.Timestamp(side["overpass"][d]) for d in cand_x]).tz_localize(None)
    ok_x = np.isfinite(np.asarray(boundary.levels(t_x), float))
    x_dates = np.array(cand_x)[ok_x]
    S_dates, S_t = np.concatenate([x_dates, S_dates]), t_x[ok_x].append(S_t)
    print(f"{EXTRA_PATH}: {len(x_dates)} screened scenes before {P['period'][0]} put before S "
          f"-> {len(S_dates)} scenes {S_dates[0]} → {S_dates[-1]}")
else:
    print("no extra extraction: the windows hold the cube's scenes only")

## 5 · The production extraction: pixels, clear mask and NDWI

`marea_extract.npz` is `marea.extract` of the cube: the pixels (more than 30 clear observations over the cube
and a wet fraction between 0.10 and 0.90), the NDWI of every scene at them (NaN where it is undefined) and the
clear mask C = SCL in {4, 5, 6, 7}. Only the rows of S stay in memory.

In [ ]:
ex = np.load(P["extract"])
keep = ex["keep"]                                      # flat indices of the pixels in the H x W frame
pos = {d: i for i, d in enumerate(ex["dates"].astype(str))}
print(f"{len(keep):,} pixels · {len(pos)} scenes · the cube's dates: {list(pos) == list(cube.dates)}")
main = [pos[d] for d in S_dates[len(x_dates):]]
Y_S, C_S = ex["Y"][main], ex["C"][main]                # (scenes of S, pixels): NDWI and clear
if len(x_dates):
    X = np.load(EXTRA_PATH)
    assert np.array_equal(X["keep"], keep), "the extra extraction is on other pixels"
    xpos = {d: i for i, d in enumerate(X["dates"].astype(str))}
    rows_x = [xpos[d] for d in x_dates]
    Y_S, C_S = np.concatenate([X["Y"][rows_x], Y_S]), np.concatenate([X["C"][rows_x], C_S])
print(f"Y and C of S: {Y_S.shape[0]} scenes × {Y_S.shape[1]:,} pixels")

The distance from the mouth and the 3-km bands of MAREA's lag fit, computed as `marea.reconstruct` computes
them (geodesic distance through water from the sea on the site's mouth side), and checked against the bands
of the production product.

In [ ]:
seeds = pit.geometry.mouth_seeds(ex["sea"], side=P["mouth_side"])
s_km = pit.geometry.along_distance(ex["sea"] | ex["inter"], seeds, 10).ravel()[keep] / 1000.0
BAND_KM = 3.0
_, centers, band_of = te.make_bands_fixed(s_km, BAND_KM, 500)
production = np.load(f"{P['marea']}/marea.npz")
print(f"{len(centers)} bands of {BAND_KM:g} km, centres {np.round(centers, 1)} km; "
      f"the production product's bands: {np.array_equal(band_of, production['band'])}")

## 6 · The truth and the pixels F

**Dutch sites.** Every pixel reads the 20 m Vaklodingen cell under it (`sample_truth` of
`experiments/v1_vaklodingen_validate.py`: the nearest cell, the first sheet with a value), year by year from
2025 down to 2023: the most recent survey with a value wins and gives the pixel its survey year. **Villaviciosa.**
The LiDAR is resampled onto the 10 m grid with `pit.reproject_to_grid`, as the production notebooks do.

The truth is then kept only inside the production no-delay product's range ± 0.25 m (a 20 m cell at the edge
of a flat often falls in the channel, which no product can represent). F is those pixels in the survey years of
the rows; no product decides it, so every method's coverage, MAREA's included, is measured on it. Above 200,000
pixels F is v19's seeded random subsample, so the pixels are v19's.

In [ ]:
import pyproj
from experiments.v0_vaklodingen_fetch import sheets_for
from experiments.v1_vaklodingen_validate import sample_truth


def vaklodingen(year):
    """{sheet: (x, y, z, year)} of the cells surveyed in `year` (a later survey of that year wins)."""
    cells = {}
    for name, _, _ in sheets_for(SITE):
        path = f"data_v4/truth/vaklodingen/{name}"
        if not os.path.exists(path):
            continue
        with xr.open_dataset(path) as ds:
            k = np.flatnonzero(ds["time"].dt.year.values == year)
            if not len(k):
                continue
            z, x, y = ds["z"].isel(time=k).values, ds["x"].values, ds["y"].values
        zt = np.full(z.shape[1:], np.nan)
        for zk in z:
            zt[np.isfinite(zk)] = zk[np.isfinite(zk)]
        cells[name] = (x, y, zt, np.full(zt.shape, year))
    return cells

In [ ]:
if SITE == "villaviciosa":
    truth = pit.reproject_to_grid(VVA_LIDAR, transform, crs, (H, W)).ravel()[keep]
    group = np.zeros(len(keep), int)                   # one row
else:
    with xr.open_dataset(P["cache"]) as ds:
        xs, ys = ds["x"].values, ds["y"].values
    to_rd = pyproj.Transformer.from_crs(crs, "EPSG:28992", always_xy=True)
    xq, yq = to_rd.transform(xs[keep % W], ys[keep // W])
    truth, group = np.full(len(keep), np.nan), np.zeros(len(keep), int)
    for year in (2025, 2024, 2023):                    # the most recent survey with a value wins
        v, _ = sample_truth(vaklodingen(year), xq, yq)
        new = np.isfinite(v) & (group == 0)
        truth[new], group[new] = v[new], year

In [ ]:
path, key = P["nodelay"]
z_nodelay = np.load(path)[key]
lo, hi = np.nanmin(z_nodelay) - RANGE_PAD_M, np.nanmax(z_nodelay) + RANGE_PAD_M
in_range = (truth >= lo) & (truth <= hi)
row_years = [row["year"] for row in ROWS]
for g in np.unique(group[np.isfinite(truth)]):
    print(f"  survey {g or 'LiDAR'}: {int((group == g).sum()):,} px with a value, "
          f"{int(((group == g) & in_range).sum()):,} inside [{lo:+.2f}, {hi:+.2f}] m"
          f" ({'a row' if g in row_years else 'left out'})")

F = np.flatnonzero(in_range & np.isin(group, row_years))
if len(F) > MAX_PX:                                    # v19's seeded subsample: the same pixels as v19
    rng = np.random.default_rng(np.random.SeedSequence([SEED, zlib.crc32(SITE.encode()) & 0x7FFFFFFF, 7]))
    F = F[np.sort(rng.permutation(len(F))[:MAX_PX])]
for row in ROWS:
    row["px"] = F[group[F] == row["year"]]
    print(f"row {row['site']:12s}: F = {len(row['px']):,} pixels")

## 7 · The windows

Window(n) is the n most recent images of S on or before the row's truth date. The windows of a row are nested
and end on the same image; a window S cannot fill is *not reachable* and says how many images it has.

In [ ]:
windows = []
for row in ROWS:
    upto = np.flatnonzero(S_dates <= row["end"])
    for n in NS:
        win = dict(site=row["site"], year=row["year"], end=row["end"], n=n, px=row["px"], available=len(upto),
                   scenes=upto[-n:] if len(upto) >= n else None)
        windows.append(win)
        if win["scenes"] is None:
            print(f"{row['site']:12s} n={n:3d}: not reachable, {len(upto)} screened images on or before {row['end']}")
        else:
            d = S_dates[win["scenes"]]
            per_year = dict(sorted(Counter(str(x)[:4] for x in d).items()))
            print(f"{row['site']:12s} n={n:3d}: {d[0]} → {d[-1]}  images per year {per_year}")

## 8 · The common inputs and the water levels

Every method of a window gets the same arrays: Y (NDWI, NaN where undefined), C (clear), W = C & (Y > 0), the
boundary level h at the scene instants without lag, the instants, the pixels and the frame. The levels
h_B(t − τ) are read once for every scene of S and every whole minute τ from −45 to 180 (MAREA's lag bank and
its per-pixel lags; τ = 0 is what every other method reads), in one boundary call over the sorted instants.

In [ ]:
shifted = {tau: S_t - pd.Timedelta(minutes=tau) for tau in range(-45, 181)}
inst = pd.DatetimeIndex(np.unique(np.concatenate([s.values for s in shifted.values()])))
lev = np.asarray(boundary.levels(inst), float)
LEV = {tau: lev[np.searchsorted(inst.values, s.values)] for tau, s in shifted.items()}
h_direct = np.asarray(boundary.levels(S_t), float)
print(f"h_B(t - tau) at {len(S_t)} scenes, tau = -45..180 min; at tau = 0 it is the level read directly "
      f"(max |diff| {np.max(np.abs(LEV[0] - h_direct)):.1e} m); range [{LEV[0].min():+.2f}, {LEV[0].max():+.2f}] m")


def make_obs(win, px):
    """The common inputs of a window at the pixels px (positions in keep)."""
    w = win["scenes"]
    Y, C = Y_S[np.ix_(w, px)], C_S[np.ix_(w, px)]
    with np.errstate(invalid="ignore"):
        wet = C & (Y > 0)                              # an undefined NDWI is never wet
    return types.SimpleNamespace(Y=Y, C=C, W=wet, h=LEV[0][w], t=S_t[w], keep=keep[px],
                                 shape=(H, W), pixel_m=10.0, site=SITE)

Flood2Topo (E5a), Tseng (E5b) and Coast X-Ray (E7) also need the water level over the whole window, not only
at the images: the boundary without lag at 1 min, built once over the longest span and sliced per window
(from 00:00 of the window's first image day to the day after its truth date).

In [ ]:
LS = None
reach = [w for w in windows if w["scenes"] is not None]
if reach and any("level_series" in fm.METHODS[m][2].get("needs", []) for m in METHODS if m in fm.METHODS):
    t0 = time.perf_counter()
    first = min(S_t[w["scenes"][0]] for w in reach).normalize()
    last = pd.Timestamp(max(w["end"] for w in reach)) + pd.Timedelta(days=1)
    LS = LevelSeries.from_boundary(boundary, first, last)
    print(f"1-min level series {first.date()} → {last.date()}: {len(LS.h):,} levels  [{time.perf_counter() - t0:.0f} s]")

## 9 · The methods

**MAREA**, as `marea.reconstruct` makes the production product (without its bootstrap interval): the lag of
every 3-km band is refitted on the window's images by the profiled Bernoulli likelihood (`te.m2a_rasch`, with
production's settings from `configs/m2.yaml` and its seeded draw of at most 1,200 pixels per band); a negative
band lag is set to 0; the profile is smoothed along the mouth distance to every pixel and rounded to the
minute; every pixel is then inverted with the level read τ minutes earlier (`marea.invert_series`), between
the window's own lowest and highest level.

In [ ]:
M2 = yaml.safe_load(open("configs/m2.yaml", encoding="utf-8"))
FIT_KW = dict(sigma0=M2["sigma0_m"], sg_grid=tuple(M2["sigma_perfil_m"]), tau_bounds=marea.TAU_BOUNDS,
              z_points=M2["z_puntos"], max_px_band=M2["max_px_banda"]["m2a"], monotone=True)


def run_marea(win, px):
    w = win["scenes"]
    bank = te.ShiftBank(marea.BANK_TAUS, [LEV[int(tau)][w] for tau in marea.BANK_TAUS])
    with np.errstate(invalid="ignore"):
        wet = C_S[w] & (Y_S[w] > 0)
    fit = te.m2a_rasch(wet, C_S[w], bank, band_of, centers, len(centers),
                       rng=np.random.default_rng(M2["seed"] + 31), **FIT_KW)
    tau_bands = np.clip(np.asarray(fit["tau"], float), 0, None)       # causality: a negative band lag is 0
    tau_px = te.smooth_lag_profile(centers, tau_bands, s_km, bandwidth_km=BAND_KM)
    tau_px = np.where(np.isfinite(tau_px), np.round(tau_px), 0.0)[px]
    h0 = LEV[0][w]
    Y0, C0 = np.nan_to_num(Y_S[np.ix_(w, px)], nan=0.0), C_S[np.ix_(w, px)]
    z = np.full(len(px), np.nan)
    for tau in np.unique(tau_px):
        cols = np.flatnonzero(tau_px == tau)
        z[cols], _ = marea.invert_series(Y0[:, cols], C0[:, cols], LEV[int(tau)][w], h0.min(), h0.max(),
                                         min_bracket=MAREA_MIN_BRACKET)
    return z, "band lags " + "/".join(f"{v:.0f}" for v in tau_bands) + " min", None

**NIDEM** as `pyintertidal.waterline_contour` implements it (the ITEM intervals of the observed tide range,
median NDWI composites, relative extents, contours tagged with the interval tides, a TIN and the confidence
filter), under the common mask SCL in {4, 5, 6, 7} instead of ITEM's. It works on whole frames, so the window's
images are read from the cube first (not timed). Its pre-flight guard (`wc.interval_coverage`: at least 1 % of
the pixels observed in every interval) is evaluated on the extraction's pixels. The value is the `filtered`
product, the validated one.

In [ ]:
cube_pos = {d: i for i, d in enumerate(cube.dates)}


def read_frames(dates):
    """NDWI and the common clear mask of whole scenes, (T, H, W), read one scene at a time."""
    ds, bands, t_dim = pit.cube.open_cube(P["cache"])
    ndwi, clear = np.empty((len(dates), H, W), np.float32), np.empty((len(dates), H, W), bool)
    try:
        for k, d in enumerate(dates):
            at = {t_dim: cube_pos[d]}
            scl = bands["SCL"].isel(at).values
            ndwi[k], _ = wc.scene_ndwi(bands["B03"].isel(at).values, bands["B08"].isel(at).values, scl)
            clear[k] = np.isin(scl, wc.CLEAR)
    finally:
        ds.close()
    return ndwi, clear


def run_nidem(win, px, frames):
    w = win["scenes"]
    j, edges, Z, U, sizes = wc.interval_assignment(LEV[0][w])
    wc.interval_coverage(wc.clear_counts_by_interval(C_S[w], Y_S[w], j)[0])   # pre-flight, on the pixels
    res = wc.nidem(wc.composites_from_arrays(*frames, j), Z, U)
    return res["filtered"].ravel()[keep[px]], f"{res['n_vertices']:,} contour vertices", None

**DEA Intertidal 2.5.3** as published, with its default parameters, is DEA's own code run in a side
interpreter (`venv-dea`) by `experiments/fixedcfg/dea_intertidal.py`, on every pixel of the extraction (its
edge cleaning needs the frame's no-data pattern). Its seconds include the side interpreter's start-up and the
file exchange; `seconds_own` is DEA's own computation as the adapter reports it.

**E4, E5a, E5b, E5c, E7 and E12** are `run(obs, cfg)` of `experiments/fixedcfg/methods.py` on the common inputs.
They are per pixel, so they run on the row's pixels F only (a pixel's value does not depend on the others).

In [ ]:
def run_dea(win, px):
    out = dea.run(make_obs(win, np.arange(len(keep))), {"dea_tmp": DEA_TMP})
    return out["z"][px], f"{out['meta']['n_candidates']:,} candidate pixels", out["meta"].get("seconds_estimator")


def run_competitor(mid, win, px):
    label, run, card = fm.METHODS[mid]
    cfg = {}
    if "level_series" in card.get("needs", []):
        start = S_t[win["scenes"][0]].normalize()
        cfg["level_series"] = LS.slice(start, pd.Timestamp(win["end"]) + pd.Timedelta(days=1))
    out = run(make_obs(win, px), cfg)
    return out["z"], out["meta"].get("reason", ""), None


def run_method(mid, win, px, frames=None):
    if mid == "MAREA":
        return run_marea(win, px)
    if mid == "NIDEM":
        return run_nidem(win, px, frames)
    if mid == "DEA":
        return run_dea(win, px)
    return run_competitor(mid, win, px)

## 10 · Scoring

The scores of a product p against the truth t on a set of pixels: RMSE = √mean((p − t)²); bias = median(p − t);
centred RMSE = the RMSE after removing each series' own median (the survey's datum is not the boundary's, so
this is the error that compares methods); the OLS slope of p on t (1 when the map keeps the relief, below 1 when
it compresses it); Pearson's r; coverage = the share of the set where p has a value. Below 30 pixels with both
values only n and the coverage are given.

In [ ]:
def scores(p, t):
    """RMSE, bias, centred RMSE, slope, r, the pixels scored and the coverage of p against t (same pixels)."""
    p, t = np.asarray(p, float), np.asarray(t, float)
    ok = np.isfinite(p) & np.isfinite(t)
    out = dict(n_px=int(ok.sum()), coverage=float(ok.mean()) if len(ok) else np.nan,
               rmse=np.nan, bias=np.nan, crmse=np.nan, slope=np.nan, r=np.nan)
    if ok.sum() >= 30:
        d = p[ok] - t[ok]
        pc, tc = p[ok] - np.median(p[ok]), t[ok] - np.median(t[ok])
        out.update(rmse=float(np.sqrt(np.mean(d ** 2))), bias=float(np.median(d)),
                   crmse=float(np.sqrt(np.mean((pc - tc) ** 2))), slope=float(np.polyfit(tc, pc, 1)[0]),
                   r=float(np.corrcoef(tc, pc)[0, 1]))
    return out

## 11 · Every window, every method

For each window, every method is called on its images and timed with `time.perf_counter` (the estimator call
alone: reading NIDEM's frames from the cube is not timed). A method that fails is recorded with its reason and
the others go on. The estimates are scored on the row's pixels F and on the common pixels (F where every
computable method of the window has a value). The CSV is rewritten after every window, so a long run leaves its
partial results.

In [ ]:
def run_window(win):
    """Every method on the window's images: the CSV rows and {method: elevation on the row's pixels F}."""
    dates = S_dates[win["scenes"]]
    base = dict(site=win["site"], survey_year=win["year"], truth_date=win["end"], n=win["n"],
                available=win["available"], first_date=dates[0], last_date=dates[-1], n_F=len(win["px"]))
    est, rows = {}, []
    for mid in METHODS:
        r = dict(base, method=mid, label=LABELS.get(mid, mid), status="ok", reason="", seconds=np.nan,
                 seconds_own=np.nan)
        try:
            frames = None
            if mid == "NIDEM":
                if any(d not in cube_pos for d in dates):
                    raise NotComputable("pre-2023 images: the extra extraction holds no whole frames")
                frames = read_frames(dates)            # the cube read, not timed
            t0 = time.perf_counter()
            z, note, own = run_method(mid, win, win["px"], frames)
            r.update(seconds=time.perf_counter() - t0, reason=note, seconds_own=np.nan if own is None else own)
            z = np.asarray(z, float)
            if np.isfinite(z).any():
                est[mid] = z
            else:
                r["status"] = "not computable"
        except (NotComputable, wc.EmptyIntervalError) as e:
            r.update(status="not computable", reason=str(e)[:300])
        except Exception as e:
            r.update(status="error", reason=repr(e)[:300])
        frames = None
        print(f"{win['site']} n={win['n']} {mid:6s} {r['status']} [{r['seconds']:.1f} s] {r['reason'][:100]}", flush=True)
        rows.append(r)
    return rows, est

In [ ]:
def score_window(win, rows, est):
    """Every computed method on the row's pixels F, and on the common pixels."""
    t_true = truth[win["px"]]
    common = np.isfinite(t_true)
    for z in est.values():
        common &= np.isfinite(z)
    for r in rows:
        r["common_share"] = float(common.mean())
        if r["method"] in est:
            z = est[r["method"]]
            r.update(scores(z, t_true))
            r.update({f"common_{k}": v for k, v in scores(z[common], t_true[common]).items() if k != "coverage"})
    return rows

In [ ]:
results = []
for win in windows:
    if win["scenes"] is None:
        results += [dict(site=win["site"], survey_year=win["year"], truth_date=win["end"], n=win["n"],
                         available=win["available"], method=m, label=LABELS.get(m, m), status="not reachable",
                         reason=f"{win['available']} screened images on or before {win['end']}") for m in METHODS]
    else:
        results += score_window(win, *run_window(win))
    pd.DataFrame(results).to_csv(f"{OUT}/{SITE}.csv", index=False)
print(f"-> {OUT}/{SITE}.csv ({len(results)} rows)")

## 12 · The table of this site

In [ ]:
df = pd.read_csv(f"{OUT}/{SITE}.csv")
cols = [m for m in METHODS if m in set(df.method)]
display(df.pivot_table(index=["site", "n"], columns="method", values="status", aggfunc="first")[cols])
ok = df[df.status == "ok"]
for score in ("crmse", "bias", "coverage", "common_crmse", "seconds"):
    if len(ok):
        print(score)
        display(ok.pivot_table(index=["site", "n"], columns="method", values=score)
                  .reindex(columns=[m for m in cols if m in set(ok.method)]).round(3))

## 13 · Error, bias and coverage against n

One figure per row (the site, or each survey-year group at the Wadden), one line per method, on the row's
pixels F. MAREA is drawn in black.

In [ ]:
COLORS = {"MAREA": "#0b0b0b", "NIDEM": "#2a78d6", "DEA": "#eb6834", "E4": "#1baf7a", "E5a": "#eda100",
          "E5b": "#e87ba4", "E5c": "#008300", "E7": "#6250d6", "E12": "#e34948"}
MARKERS = {"MAREA": "o", "NIDEM": "s", "DEA": "D", "E4": "^", "E5a": "v", "E5b": "<", "E5c": ">", "E7": "P",
           "E12": "X"}
for site_row, d in ok.groupby("site"):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
    for mid in cols:
        m = d[d.method == mid].sort_values("n")
        for ax, score in zip(axes, ("crmse", "bias", "coverage")):
            ax.plot(m.n, m[score], marker=MARKERS.get(mid, "o"), ms=7, lw=2.6 if mid == "MAREA" else 1.6,
                    color=COLORS.get(mid), label=LABELS.get(mid, mid))
    for ax, title in zip(axes, ("centred RMSE (m)", "bias = median(p − t) (m)", "coverage of F")):
        ax.set_title(title, loc="left")
        ax.set_xlabel("images in the window, n")
        ax.set_xticks(sorted(d.n.unique()))
        ax.grid(alpha=0.3)
    axes[1].axhline(0, color="0.5", lw=0.8)
    axes[2].legend(fontsize=8, loc="lower right")
    fig.suptitle(f"{site_row}: every method against the number of images", x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(f"{OUT}/{site_row}_curves.png", dpi=130)

## 14 · Provenance, and the check against v19

The commit and the files that differ from it, the package versions and the host. Then, when
`experiments/v19_fixed_config_tables.py` has written its plan or its run for this site, its windows must hold
the same images as these, and its MAREA column (quality rule `MAREA_MIN_BRACKET`, each method on its own pixels
of F) must give the same scores.

In [ ]:
import platform, socket, subprocess
import scipy
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.rstrip()
dirty = [l for l in git("status", "--porcelain").splitlines() if not l.startswith("??")]
print("commit  ", git("rev-parse", "HEAD"), git("rev-parse", "--abbrev-ref", "HEAD"))
print(f"modified {len(dirty)} tracked file(s)" + "".join(f"\n  {l}" for l in dirty[:15]))
print("host    ", socket.gethostname(), "· OMP_NUM_THREADS", os.environ.get("OMP_NUM_THREADS"))
print("versions", f"python {platform.python_version()} · numpy {np.__version__} · pandas {pd.__version__} · "
      f"xarray {xr.__version__} · scipy {scipy.__version__} · pyintertidal {pit.__version__}")

In [ ]:
plan = next((f for f in (f"{V19_OUT}/{SITE}_windows.json", f"{V19_OUT}/{SITE}_plan.json") if os.path.exists(f)), None)
if plan:
    v19 = {(w.get("group", ROWS[0]["year"]), w["n"]): w["dates"] for w in json.load(open(plan))["windows"]
           if w.get("reachable") and not w.get("gate_only")}
    same = [list(S_dates[w["scenes"]]) == v19.get((w["year"], w["n"])) for w in windows if w["scenes"] is not None]
    print(f"windows: {sum(same)} of {len(same)} hold the same images as v19's ({plan})")
else:
    print(f"no v19 plan or run in {V19_OUT}: the windows are not compared")

csv19 = f"{V19_OUT}/{SITE}.csv"
if os.path.exists(csv19):
    v = pd.read_csv(csv19)
    v = v[(v.method == "E0b") & (v.variant == f"mb{MAREA_MIN_BRACKET}") & (v.scoring == "own") & (v.status == "ok")]
    v = v.rename(columns={"n_scenes": "n", "rmse": "crmse_v19", "bias": "bias_v19", "coverage": "coverage_v19"})
    m = df[(df.method == "MAREA") & (df.status == "ok")].merge(v[["site", "n", "crmse_v19", "bias_v19", "coverage_v19"]],
                                                                on=["site", "n"])
    display(m[["site", "n", "crmse", "crmse_v19", "bias", "bias_v19", "coverage", "coverage_v19"]])
    print("MAREA vs v19, max |difference|:",
          {k: float(np.max(np.abs(m[k] - m[f"{k}_v19"]))) if len(m) else None for k in ("crmse", "bias", "coverage")})
else:
    print(f"no v19 results in {csv19}: the MAREA column is not compared")